In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import time

from tqdm import tqdm

In [ ]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
df = pd.read_csv("labeled_logs_streamed_tbird_v1.csv")



In [4]:
# Map EventIds to integers
event_vocab = {eid: idx for idx, eid in enumerate(sorted(df["EventId"].unique()))}
df["EventIdx"] = df["EventId"].map(event_vocab)


In [5]:
block_size = 500
sequence_length = 80


In [6]:
block_starts = list(range(0, len(df), block_size))
block_labels = [
    1 if df.iloc[i:i+block_size]["Label"].sum() > 0 else 0
    for i in block_starts
]



In [8]:
from sklearn.model_selection import train_test_split

train_block_starts, test_block_starts = train_test_split(
    block_starts, test_size=0.2, stratify=block_labels, random_state=42
)



In [9]:
class TBirdSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, block_starts, sequence_length=80, label_column="Label"):
        self.df = df
        self.block_starts = block_starts
        self.sequence_length = sequence_length
        self.label_column = label_column
        self.samples = []

        # Precompute valid sequence start indices within each block
        for start in block_starts:
            block = df.iloc[start:start + block_size].reset_index(drop=True)
            block["EventIdx"] = block["EventId"].map(event_vocab)

            for i in range(0, len(block) - sequence_length):
                seq = block["EventIdx"].iloc[i:i + sequence_length].tolist()
                labels = block[label_column].iloc[i:i + sequence_length].tolist()
                label = 1 if 1 in labels else 0
                self.samples.append((seq, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, label = self.samples[idx]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [10]:
train_dataset = TBirdSequenceDataset(df, train_block_starts, sequence_length)


In [11]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)



In [12]:
class EventSequenceModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dropout=.3),
            num_layers=2
        )
        self.attn = nn.Linear(embed_dim, 1)
        
        #self.fc = nn.Linear(embed_dim, 2)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(.3),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        batch_size, seq_len = x.size()
        
        x = self.embed(x)  # [batch, seq_len, embed_dim]

        # Add positional encoding (learned embedding)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        pos_emb = self.pos_encoder(positions)  # [batch, seq_len, embed_dim]
        x = x + pos_emb
        
        x = x.permute(1, 0, 2)  # [seq_len, batch, embed_dim]
        x = self.transformer(x)
        #x = x[-1]  # last token
        #x = x.mean(dim=0)
        attn_weights = torch.softmax(self.attn(x), dim=0)  # [seq_len, batch, 1]
        x = (x * attn_weights).sum(dim=0)  # weighted sum
        
        return self.fc(x)


In [13]:
model = EventSequenceModel(vocab_size=len(event_vocab), embed_dim=64, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)



C:\Users\pro\.conda\envs\logenv\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [14]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)
    for batch_x, batch_y in loop:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = output.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        
    avg_loss = total_loss / len(train_loader.dataset)
    train_acc = correct / len(train_loader.dataset)

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")
    print(f"Epoch {epoch+1} Loss: {avg_loss:.6f}")
    print(f"Epoch {epoch+1} Accuracy: {train_acc:.5f}")



Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 192183/192183 [50:37<00:00, 63.28it/s]


Epoch 1 Loss: 2226.4501
Epoch 1 Loss: 0.000181
Epoch 1 Accuracy: 0.99695


Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 192183/192183 [50:52<00:00, 62.97it/s]


Epoch 2 Loss: 1778.1462
Epoch 2 Loss: 0.000145
Epoch 2 Accuracy: 0.99745


Epoch 3: 100%|█████████████████████████████████████████████████████████████████| 192183/192183 [49:14<00:00, 65.04it/s]


Epoch 3 Loss: 1745.2576
Epoch 3 Loss: 0.000142
Epoch 3 Accuracy: 0.99747


Epoch 4: 100%|█████████████████████████████████████████████████████████████████| 192183/192183 [49:04<00:00, 65.28it/s]


Epoch 4 Loss: 1692.0914
Epoch 4 Loss: 0.000138
Epoch 4 Accuracy: 0.99755


Epoch 5: 100%|█████████████████████████████████████████████████████████████████| 192183/192183 [49:03<00:00, 65.29it/s]

Epoch 5 Loss: 1659.4262
Epoch 5 Loss: 0.000135
Epoch 5 Accuracy: 0.99758


In [ ]:
# Shipped checkpoint: models/TableIV_TableVII_Fig8_tbird_classifier.pth
# (retraining overwrites it with your own run)
torch.save(model.state_dict(), "./models/TableIV_TableVII_Fig8_tbird_classifier.pth")

In [16]:
test_dataset = TBirdSequenceDataset(df, test_block_starts, sequence_length)